In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# Parameters
G = 14
p_values = [0.75, 0.85]
N_e = 728

In [3]:
# Recomb map file name pattern
ceu_pattern = "data/pyrho_CEU_recomb_map_hapmap_format_hg38_chr_{}.txt"
yri_pattern = "data/pyrho_YRI_recomb_map_hapmap_format_hg38_chr_{}.txt"

In [4]:
# Chrom lengths (autosomes)
chrom_len_path = "data/chrom_len.csv"
chrom_len_df = pd.read_csv(chrom_len_path)

chrom_len_df["filtered_length_chrom_bp"] = chrom_len_df["filtered_length_chrom_bp"].astype(int)

In [5]:
chrom_len_df

,chrom_number,full_length_chrom_bp,removed_bp,filtered_length_chrom_bp
0,1,248956422,69770177,179186245
1,2,242193529,46203029,195990500
2,3,198295559,35028091,163267468
3,4,190214555,33800073,156414482
4,5,181538259,35180560,146357699
5,6,170805979,31540776,139265203
6,7,159345973,37697915,121648058
7,8,145138636,26252797,118885839
8,9,138394717,48449484,89945233
9,10,133797422,27688176,106109246


In [6]:
out_dir = "out"
os.makedirs(out_dir, exist_ok=True)

In [7]:
# Helper function that loads maps to df
def load_map(path):
    df = pd.read_csv(path, sep="\t", comment="#", header=None)

    # If the first row is a header (e.g. Position(bp)), drop it
    if isinstance(df.iloc[0, 1], str) and "Position" in df.iloc[0, 1]:
        df = df.iloc[1:].reset_index(drop=True)

    df = df.iloc[:, :4].copy()
    df.columns = ["chrom", "position", "rate_cM_Mb", "genetic_map"]

    df["position"] = df["position"].astype(int)
    df["rate_cM_Mb"] = df["rate_cM_Mb"].astype(float)
    df["genetic_map"] = df["genetic_map"].astype(float)

    return df.sort_values("position").reset_index(drop=True)

In [8]:
# Function that adds starting position to maps
def add_pos0(df):
    if (df["position"] == 0).any():
        return df
    return pd.concat([
        df,
        pd.DataFrame([{
            "chrom": df.iloc[0]["chrom"],
            "position": 0,
            "rate_cM_Mb": 0.0,
            "genetic_map": 0.0
        }])
    ]).sort_values("position").reset_index(drop=True)

In [9]:
# Function formats maps for calculation
def build_recomb_df(ceu_df, yri_df):
    ceu_df = add_pos0(ceu_df)
    yri_df = add_pos0(yri_df)

    # Union of unique positions
    positions = sorted(set(ceu_df["position"]) | set(yri_df["position"]))

    # Fill missing positions
    ceu = ceu_df.set_index("position").reindex(positions, method="ffill").reset_index()
    yri = yri_df.set_index("position").reindex(positions, method="ffill").reset_index()

    #Return df and convert cM/Mb → Morgans/bp
    return pd.DataFrame({
        "position": positions,
        "r_a1": ceu["rate_cM_Mb"] * 1e-8,
        "r_a2": yri["rate_cM_Mb"] * 1e-8
    })

In [10]:
# Function that calculates theoretical switch count for all autosomes at gen 14
# Gen 14 is our estimated time since admixture for African American sample (read methods for more detail)
def expected_switches_g14(recomb_df, chr_len, p):
    x = recomb_df["position"].to_numpy()
    r = recomb_df["r_a1"].to_numpy() + recomb_df["r_a2"].to_numpy()

    delta_x = np.diff(x)
    delta_x = np.append(delta_x, chr_len - x[-1])

    integrand = r * p * (1 - p)
    integrand = integrand[:len(delta_x)]

    decay = 2 * N_e * (1 - (1 - 1 / (2 * N_e)) ** G)

    return decay * np.sum(delta_x * integrand)

In [11]:
results = {p: [] for p in p_values}

for _, row in chrom_len_df.iterrows():
    chrom = int(row["chrom_number"])
    chr_len = int(row["filtered_length_chrom_bp"])

    ceu_df = load_map(ceu_pattern.format(chrom))
    yri_df = load_map(yri_pattern.format(chrom))

    recomb_df = build_recomb_df(ceu_df, yri_df)

    for p in p_values:
        results[p].append(
            expected_switches_g14(recomb_df, chr_len, p)
        )

/var/folders/5c/nxn1ystj29n4gjk0bm30r4r80000gn/T/ipykernel_36767/1636244425.py:3: DtypeWarning: Columns (1,2,3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, sep="\t", comment="#", header=None)
/var/folders/5c/nxn1ystj29n4gjk0bm30r4r80000gn/T/ipykernel_36767/1636244425.py:3: DtypeWarning: Columns (1,2,3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, sep="\t", comment="#", header=None)


In [12]:
for p in p_values:
    chrom_len_df[f"expected_switches_g14_p{p}"] = results[p]

chrom_len_df.to_csv(
    os.path.join(out_dir, "chrom_lengths_with_expected_switches_g14_filtered_len.csv"),
    index=False
)

chrom_len_df

,chrom_number,full_length_chrom_bp,removed_bp,filtered_length_chrom_bp,expected_switches_g14_p0.75,expected_switches_g14_p0.85
0,1,248956422,69770177,179186245,9.626250,6.545850
1,2,242193529,46203029,195990500,9.237231,6.281317
2,3,198295559,35028091,163267468,7.699202,5.235457
3,4,190214555,33800073,156414482,8.314977,5.654184
4,5,181538259,35180560,146357699,7.108782,4.833972
5,6,170805979,31540776,139265203,6.786088,4.614540
6,7,159345973,37697915,121648058,6.792215,4.618706
7,8,145138636,26252797,118885839,7.061676,4.801940
8,9,138394717,48449484,89945233,7.652137,5.203453
9,10,133797422,27688176,106109246,6.197492,4.214295


In [13]:
# Adding empirical switch count estimates
emp_path = "data/ASW_empir_switch_count.csv"
emp_df = pd.read_csv(emp_path)

In [14]:
empirical_path = "data/ASW_empir_switch_count.csv"
emp_df = pd.read_csv(empirical_path)

# Rename chrom column to match theory df
emp_df = emp_df.rename(columns={"chr": "chrom_number"})

In [15]:
# Merge and format df
merged_df = chrom_len_df.merge(
    emp_df[
        [
            "chrom_number",
            "boot_mean",
            "95_lower_ci",
            "95_upper_ci",
        ]
    ],
    on="chrom_number",
    how="left"
)

merged_df["empirical_switches"] = (
    merged_df["boot_mean"].map("{:.3f}".format)
    + " ("
    + merged_df["95_lower_ci"].map("{:.3f}".format)
    + "–"
    + merged_df["95_upper_ci"].map("{:.3f}".format)
    + ")"
)

merged_df["theoretical_switches_p0.75"] = (
    merged_df["expected_switches_g14_p0.75"]
    .map("{:.3f}".format)
)

merged_df["theoretical_switches_p0.85"] = (
    merged_df["expected_switches_g14_p0.85"]
    .map("{:.3f}".format)
)

merged_df = merged_df.drop(
    columns=["boot_mean", "95_lower_ci", "95_upper_ci", "expected_switches_g14_p0.75", "expected_switches_g14_p0.85", "full_length_chrom_bp", "removed_bp"]
)

In [16]:
merged_df

,chrom_number,filtered_length_chrom_bp,empirical_switches,theoretical_switches_p0.75,theoretical_switches_p0.85
0,1,179186245,6.142 (5.213–7.117),9.626,6.546
1,2,195990500,4.880 (4.159–5.790),9.237,6.281
2,3,163267468,4.236 (3.548–5.007),7.699,5.235
3,4,156414482,3.539 (2.881–4.292),8.315,5.654
4,5,146357699,4.013 (3.252–4.830),7.109,4.834
5,6,139265203,3.394 (2.837–3.990),6.786,4.615
6,7,121648058,3.214 (2.720–3.780),6.792,4.619
7,8,118885839,2.659 (2.156–3.250),7.062,4.802
8,9,89945233,3.254 (2.745–3.739),7.652,5.203
9,10,106109246,3.114 (2.620–3.641),6.197,4.214
